# 전처리 코드 통합본

## 테이블 목록
| 테이블명 | 파일명 |
|---|---|
| 재무테이블_clean1 | finance_clean1.parquet |
| 재무테이블_clean2 | finance_clean2.parquet |
| 유저 일자 활동 | tb_user_daily_activity.parquet |
| 광고 운영 캘린더 | sched.parquet |
| 광고성과 | ads_outcome.parquet |
| 메인 퍼널 | main_funnel.parquet |
| 클릭-완료 팩트 | fact_click_reward.parquet |
| 광고참여목록 정제 | ads_join_info_labeled.parquet |

## 기준 날짜 사용 원칙
- `click_date` : 클릭 수, DAU, 재방문, 행동 관찰 지표에 사용
- `regdate` : 완료 수, 적립 처리 시점, 완료 기준 성과, 마진 발생 시점에 사용
- 같은 테이블 안에 두 날짜를 둘 수 있지만 해석 기준은 반드시 분리

## 피드백 반영 목록
- **[1]** `strict=False` 파싱 실패 건수 검증 추가
- **[2]** 매체 분류 컬럼 `df` → `df_list` 반영 (join으로 명시적 반영)
- **[3]** 원본 유지 + 이상치 플래그화 (직접 삭제 → `flag_invalid_*` 플래그 세우기)
- **[4]** 행동 유형 컬럼 값 체계 통일 (복합행동/미확정 → 숫자 코드: 0/99)
- **[5]** 전처리 완료 확인 강화 (PK 중복, null, 음수, 플래그 분포)
- **[버그수정]** `ads_action_diff_flag` 계산 시 `action_confidence` 미정의 참조 오류 수정

## 추가 반영
- `전처리파트 다듬기` 파일에만 있던 `ads_join_info_labeled.parquet` 로드/형변환/EDA-ready 저장을 추가했다.
- `df_ji_labeled`는 광고참여목록 정제 테이블로, 기존 날짜 파생 컬럼(`click_date_only`, `click_hour`, `click_weekday`, `reg_date_only`, `reg_hour`, `reg_weekday`)을 보존해 저장한다.



---
## 0. 라이브러리 로드

In [7]:
import polars as pl
from pathlib import Path

# 전처리 완료 파일 경로 (parquet_debug01 폴더)
DATA_PATH = r'C:/Users/hoo58/OneDrive/바탕 화면/tables/DATA/parquet_debug01/'

# EDA-ready 저장 경로
EDA_READY_DIR = Path(r'C:/Users/hoo58/OneDrive/바탕 화면/tables/DATA/eda_ready_new')

# 원본 parquet 경로
PARQUET_PATH = r'C:/Users/hoo58/OneDrive/바탕 화면/tables/DATA/parquet_debug/'

---
## 공통 유틸

In [8]:
# strict=False 파싱 후 null 발생 건수 검증 함수
# 파싱 실패 건이 조용히 null로 저장되는 걸 방지
def check_parse_nulls(df: pl.DataFrame, cols: list[str], label: str):
    print(f"\n── [{label}] 날짜 파싱 null 검증 ──")
    for col in cols:
        if col not in df.columns:
            continue
        n_null = df[col].is_null().sum()
        flag = "⚠️  주의!" if n_null > 0 else "✅"
        print(f"  {flag}  {col}: null {n_null:,}건")

weekday_map_str = {'1':'Mon','2':'Tue','3':'Wed','4':'Thu','5':'Fri','6':'Sat','7':'Sun'}
weekday_map_int = {1:'Mon', 2:'Tue', 3:'Wed', 4:'Thu', 5:'Fri', 6:'Sat', 7:'Sun'}

---
## 1. 재무 테이블

### 테이블 설명
- `finance_basic` : 적립(rwd_idx) 1건 = 1행. 클릭-적립 연결 로그와 비용 정보
- `finance_list` : 광고별 재무 정보

### 전처리 내용
1. 날짜/시간 파생 컬럼 생성 (click_date, regdate 기준)
2. 요일 파생 컬럼 생성
3. ive_margin 명시 생성 (adv_cost - earn_cost)
4. 특수 마진 그룹 플래그 생성
5. 마진 분석용 정상 데이터 분리
6. 특수 그룹 외 음수 마진 건수 검증

In [9]:
finance_clean1 = pl.read_parquet(DATA_PATH + 'finance_clean1.parquet')
finance_clean2  = pl.read_parquet(DATA_PATH + 'finance_clean2.parquet')

print('finance_clean1 shape:', finance_clean1.shape)
print('finance_clean2 shape :', finance_clean2.shape)

finance_clean1 shape: (1474776, 19)
finance_clean2 shape : (1383784, 19)


In [10]:
finance_clean1 = pl.read_parquet(DATA_PATH + 'finance_clean1.parquet')
finance_clean2  = pl.read_parquet(DATA_PATH + 'finance_clean2.parquet')

print('finance_clean1 shape:', finance_clean1.shape)
print('finance_clean2 shape :', finance_clean2.shape)

finance_clean1 shape: (1474776, 19)
finance_clean2 shape : (1383784, 19)


In [11]:
# 날짜/시간 파생 컬럼 생성
for df_name, df_obj in [('finance_clean1', finance_clean1), ('finance_clean2', finance_clean2)]:
    df_obj = df_obj.with_columns([
        pl.col('click_date').dt.date().alias('click_date_only'),
        pl.col('click_date').dt.hour().cast(pl.Int8).alias('click_hour'),
        pl.col('click_date').dt.time().alias('click_time'),
        pl.col('regdate').dt.date().alias('reg_date_only'),
        pl.col('regdate').dt.hour().cast(pl.Int8).alias('reg_hour'),
        pl.col('regdate').dt.time().alias('reg_time'),
    ])
    if df_name == 'finance_clean1':
        finance_clean1 = df_obj
    else:
        finance_clean2 = df_obj

# 요일 생성
finance_clean1 = finance_clean1.with_columns([
    pl.col('click_date').dt.weekday().cast(pl.String).replace(weekday_map_str).alias('click_weekday'),
    pl.col('regdate').dt.weekday().cast(pl.String).replace(weekday_map_str).alias('reg_weekday'),
])
finance_clean2 = finance_clean2.with_columns([
    pl.col('click_date').dt.weekday().cast(pl.String).replace(weekday_map_str).alias('click_weekday'),
    pl.col('regdate').dt.weekday().cast(pl.String).replace(weekday_map_str).alias('reg_weekday'),
])

In [12]:
# ive_margin 명시 생성 및 특수 마진 그룹 식별
# 아이브 자체 퀘스트 광고 (ads_type=9, mda_idx=270, adv_cost=0, show_cost=0)
finance_clean1 = finance_clean1.with_columns([
    (pl.col('adv_cost') - pl.col('earn_cost')).alias('ive_margin'),
    (
        (pl.col('ads_type') == 9) &
        (pl.col('ads_save_way') == '모든 서브퀘스트 완료') &
        (pl.col('mda_idx') == 270) &
        (pl.col('adv_cost') == 0) &
        (pl.col('show_cost') == 0)
    ).alias('flag_special_margin_group')
])
finance_clean2 = finance_clean2.with_columns([
    (pl.col('adv_cost') - pl.col('earn_cost')).alias('ive_margin')
])

# 마진 분석용 정상 데이터 분리 (원본 finance_list 유지)
finance_margin = finance_clean1.filter(~pl.col('flag_special_margin_group'))

negative_margin_cnt = finance_margin.filter(pl.col('ive_margin') < 0).height
print('전체 행수                :', finance_clean1.height)
print('특수 마진 그룹           :', int(finance_clean1['flag_special_margin_group'].sum()))
print('정상 마진 분석 행수       :', finance_margin.height)
print('특수 그룹 외 음수 마진 건수:', negative_margin_cnt)

전체 행수                : 1474776
특수 마진 그룹           : 3528
정상 마진 분석 행수       : 1471248
특수 그룹 외 음수 마진 건수: 0


---
## 2. 원본 테이블 형변환 및 파생 컬럼

### 테이블 설명
- `df_rpt` : 시간대별 광고 보고서
- `df_join_info` : 광고 참여 정보
- `df_rwd` : 광고 적립 정보
- `df_list` : 광고 목록

### 전처리 내용
1. 각 테이블 날짜/시간 형변환 + **[피드백 1]** 파싱 실패 검증
2. df_list 매체 분류 파생 컬럼 생성 + **[피드백 2]** df_list에 명시적 반영
3. df_list 행동 유형 파생 컬럼 생성 + **[피드백 4]** 값 체계 숫자 통일

In [13]:
df_rpt       = pl.read_parquet(PARQUET_PATH + '아이브시간대별광고리포트_1년치__Result *.parquet')
df_join_info = pl.read_parquet(PARQUET_PATH + 'IVE_광고참여정보__Result *.parquet')
df_rwd       = pl.read_parquet(PARQUET_PATH + 'IVE_광고적립__Result *.parquet')
df_list      = pl.read_parquet(PARQUET_PATH + 'IVE_광고목록__Result 1.parquet')

print('df_rpt shape      :', df_rpt.shape)
print('df_join_info shape:', df_join_info.shape)
print('df_rwd shape      :', df_rwd.shape)
print('df_list shape     :', df_list.shape)

# 기대 shape
# df_rpt       : (6953146, 10)
# df_join_info : (16854865, 17)
# df_rwd       : (1475031, 15)
# df_list      : (445260, 30)

df_rpt shape      : (6953146, 10)
df_join_info shape: (16854865, 17)
df_rwd shape      : (1475031, 15)
df_list shape     : (445260, 30)


In [14]:
# df_rpt 형변환
df_rpt = df_rpt.with_columns([
    pl.col('rpt_time_date').str.replace_all('"', '').str.to_date('%Y-%m-%d'),
    pl.col('rpt_time_time').cast(pl.Int8),
])

# df_join_info 형변환
df_join_info = df_join_info.with_columns([
    pl.col('click_day').str.replace_all('"', '').str.to_date('%Y-%m-%d'),
    pl.col('click_time').cast(pl.Int8),
    pl.col('click_date').str.replace_all('"', '').str.to_datetime(format='%Y-%m-%d %H:%M:%S', strict=False),
    pl.col('exp_day').str.replace_all('"', '').str.to_date('%Y-%m-%d'),
    pl.col('network').cast(pl.Utf8).cast(pl.Categorical),
])

# df_rwd 형변환
df_rwd = df_rwd.with_columns([
    pl.col('click_date').str.replace_all('"', '').str.to_datetime(format='%Y-%m-%d %H:%M:%S', strict=False),
    pl.col('regdate').str.replace_all('"', '').str.to_datetime(format='%Y-%m-%d %H:%M:%S', strict=False),
])

# df_list 형변환
df_list = df_list.with_columns([
    pl.col('ads_type').cast(pl.Utf8).cast(pl.Categorical),
    pl.col('ads_category').cast(pl.Utf8).cast(pl.Categorical),
    pl.when(pl.col('ads_day_cap') == 'Y').then(True)
      .when(pl.col('ads_day_cap') == 'N').then(False)
      .otherwise(None).alias('ads_day_cap'),
    pl.col('ads_sdate').str.replace_all('"', '').str.to_datetime(format='%Y-%m-%d %H:%M:%S', strict=False),
    pl.col('ads_edate').str.replace_all('"', '').str.to_datetime(format='%Y-%m-%d %H:%M:%S', strict=False),
    pl.col('ads_os_type').cast(pl.Utf8).cast(pl.Categorical),
    pl.col('ads_rejoin_type').cast(pl.Utf8).cast(pl.Categorical),
    pl.when(pl.col('ads_require_adid') == 'Y').then(True)
      .when(pl.col('ads_require_adid') == 'N').then(False)
      .otherwise(None).alias('ads_require_adid'),
    pl.when(pl.col('delyn') == 'Y').then(True)
      .when(pl.col('delyn') == 'N').then(False)
      .otherwise(None).alias('delyn'),
    pl.col('regdate').str.replace_all('"', '').str.to_datetime(format='%Y-%m-%d %H:%M:%S', strict=False),
])

print('형변환 완료')

형변환 완료


In [15]:
# strict=False 파싱 실패 검증
check_parse_nulls(df_join_info, ['click_date'], 'df_join_info')
check_parse_nulls(df_rwd,       ['click_date', 'regdate'], 'df_rwd')
check_parse_nulls(df_list,      ['ads_sdate', 'ads_edate', 'regdate'], 'df_list')


── [df_join_info] 날짜 파싱 null 검증 ──
  ✅  click_date: null 0건

── [df_rwd] 날짜 파싱 null 검증 ──
  ✅  click_date: null 0건
  ✅  regdate: null 0건

── [df_list] 날짜 파싱 null 검증 ──
  ⚠️  주의!  ads_sdate: null 383건
  ⚠️  주의!  ads_edate: null 383건
  ✅  regdate: null 0건


### 매체 분류 파생 컬럼 생성

In [16]:
# df = df_list.clone() 후 파생 컬럼을 만들고
#            마지막에 df_list에 join하여 반영 (기존엔 df에만 만들고 끝냈음)
df = df_list.clone()

# 1단계: 통합 텍스트
df = df.with_columns(
    pl.concat_str([
        pl.col('ads_name').fill_null(''),
        pl.lit(' '),
        pl.col('ads_summary').fill_null(''),
        pl.lit(' '),
        pl.col('ads_guide').fill_null('')
    ]).str.to_lowercase().alias('text_all')
)

# 2단계: mention flag (매체명이 텍스트에 등장하기만 해도 1)
df = df.with_columns([
    pl.col('text_all').str.contains(r'인스타그램|인스타|instagram|insta').cast(pl.Int8).alias('media_instagram_mention'),
    pl.col('text_all').str.contains(r'네이버|naver').cast(pl.Int8).alias('media_naver_mention'),
    pl.col('text_all').str.contains(r'유튜브|유투브|youtube').cast(pl.Int8).alias('media_youtube_mention'),
    pl.col('text_all').str.contains(r'페이스북|페북|facebook').cast(pl.Int8).alias('media_facebook_mention'),
    pl.col('text_all').str.contains(r'트위터|twitter|엑스|x\.com').cast(pl.Int8).alias('media_twitter_mention'),
    pl.col('text_all').str.contains(r'11번가|11st').cast(pl.Int8).alias('media_11st_mention'),
    pl.col('text_all').str.contains(r'쿠팡|coupang').cast(pl.Int8).alias('media_coupang_mention'),
])

In [17]:
# 3단계: target / info_only flag
ACTION_PATTERN = r'(방문|접속|이동|클릭|검색|팔로우|구독|좋아요|조회|시청|보기|확인|입력|참여|응모|작성|다운로드|설치|구매|주문|리뷰|후기|공유|가입)'
INFO_PATTERN   = r'(공식|공식홈페이지|홈페이지|공식페이지|공식계정|커뮤니티|개발자|고객센터|문의|제휴|제작사|퍼블리셔|운영|카페|블로그|사이트|참고|안내)'

MEDIA_PATTERNS = {
    'instagram': r'인스타그램|인스타|instagram|insta',
    'naver':     r'네이버|naver',
    'youtube':   r'유튜브|유투브|youtube',
    'facebook':  r'페이스북|페북|facebook',
    'twitter':   r'트위터|twitter|엑스|x\.com',
    '11st':      r'11번가|11st',
    'coupang':   r'쿠팡|coupang',
}

target_cols    = []
info_only_cols = []

for media, pat in MEDIA_PATTERNS.items():
    t_col  = f'media_{media}_target'
    io_col = f'media_{media}_info_only'
    df = df.with_columns([
        (
            pl.col('text_all').str.contains(rf'({pat}).{{0,12}}{ACTION_PATTERN}') |
            pl.col('text_all').str.contains(rf'{ACTION_PATTERN}.{{0,15}}({pat})')
        ).cast(pl.Int8).alias(t_col),
        (
            pl.col('text_all').str.contains(rf'({pat}).{{0,12}}{INFO_PATTERN}') |
            pl.col('text_all').str.contains(rf'{INFO_PATTERN}.{{0,12}}({pat})')
        ).cast(pl.Int8).alias(io_col),
    ])
    target_cols.append(t_col)
    info_only_cols.append(io_col)

mention_cols = [f'media_{m}_mention' for m in MEDIA_PATTERNS]

# 4단계: 집계 컬럼
df = df.with_columns([
    sum(pl.col(c) for c in mention_cols).alias('mentioned_media_cnt'),
    sum(pl.col(c) for c in target_cols).alias('target_media_cnt'),
    sum(pl.col(c) for c in info_only_cols).alias('info_only_media_cnt'),
])

In [18]:
# 5단계: 채널 분류
df = df.with_columns([
    pl.when(pl.col('target_media_cnt') >= 2).then(pl.lit('복수매체'))
    .when((pl.col('target_media_cnt') == 1) & (pl.col('media_instagram_target') == 1)).then(pl.lit('인스타'))
    .when((pl.col('target_media_cnt') == 1) & (pl.col('media_naver_target') == 1)).then(pl.lit('네이버'))
    .when((pl.col('target_media_cnt') == 1) & (pl.col('media_youtube_target') == 1)).then(pl.lit('유튜브'))
    .when((pl.col('target_media_cnt') == 1) & (pl.col('media_facebook_target') == 1)).then(pl.lit('페이스북'))
    .when((pl.col('target_media_cnt') == 1) & (pl.col('media_twitter_target') == 1)).then(pl.lit('트위터'))
    .when((pl.col('target_media_cnt') == 1) & (pl.col('media_11st_target') == 1)).then(pl.lit('11번가'))
    .when((pl.col('target_media_cnt') == 1) & (pl.col('media_coupang_target') == 1)).then(pl.lit('쿠팡'))
    .when(
        (pl.col('mentioned_media_cnt') >= 1) &
        (pl.col('target_media_cnt') == 0) &
        (pl.col('info_only_media_cnt') >= 1)
    ).then(pl.lit('단순언급/미확정'))
    .otherwise(pl.lit('미확정'))
    .alias('to_channel_rule')
])

# 6단계: 신뢰도 / 검수 플래그
df = df.with_columns([
    pl.when(pl.col('target_media_cnt') == 1).then(pl.lit('HIGH'))
    .when(pl.col('target_media_cnt') >= 2).then(pl.lit('MULTI_TARGET'))
    .when(
        (pl.col('mentioned_media_cnt') >= 1) &
        (pl.col('target_media_cnt') == 0) &
        (pl.col('info_only_media_cnt') >= 1)
    ).then(pl.lit('LOW_MENTION_ONLY'))
    .when((pl.col('mentioned_media_cnt') >= 1) & (pl.col('target_media_cnt') == 0))
    .then(pl.lit('REVIEW'))
    .otherwise(pl.lit('NO_MEDIA_MENTION'))
    .alias('media_confidence'),

    pl.when(pl.col('target_media_cnt') >= 2).then(pl.lit(1))
    .when(
        (pl.col('mentioned_media_cnt') >= 1) &
        (pl.col('target_media_cnt') == 0) &
        (pl.col('info_only_media_cnt') == 0)
    ).then(pl.lit(1))
    .otherwise(pl.lit(0))
    .cast(pl.Int8).alias('media_review_flag')
])

# df → df_list 반영 (join)
media_cols = (
    ['ads_idx', 'text_all'] +
    mention_cols + target_cols + info_only_cols +
    ['mentioned_media_cnt', 'target_media_cnt', 'info_only_media_cnt',
     'to_channel_rule', 'media_confidence', 'media_review_flag']
)
df_list = df_list.join(df.select(media_cols), on='ads_idx', how='left')
print('매체 분류 컬럼이 df_list에 반영되었습니다.')
print(df_list.select('to_channel_rule', 'media_confidence')
             .group_by(['to_channel_rule', 'media_confidence'])
             .len().sort('len', descending=True))

매체 분류 컬럼이 df_list에 반영되었습니다.
shape: (11, 3)
┌─────────────────┬──────────────────┬────────┐
│ to_channel_rule ┆ media_confidence ┆ len    │
│ ---             ┆ ---              ┆ ---    │
│ str             ┆ str              ┆ u32    │
╞═════════════════╪══════════════════╪════════╡
│ 미확정          ┆ NO_MEDIA_MENTION ┆ 429857 │
│ 네이버          ┆ HIGH             ┆ 6595   │
│ 미확정          ┆ REVIEW           ┆ 6426   │
│ 트위터          ┆ HIGH             ┆ 730    │
│ 유튜브          ┆ HIGH             ┆ 690    │
│ …               ┆ …                ┆ …      │
│ 페이스북        ┆ HIGH             ┆ 226    │
│ 단순언급/미확정 ┆ LOW_MENTION_ONLY ┆ 109    │
│ 11번가          ┆ HIGH             ┆ 99     │
│ 쿠팡            ┆ HIGH             ┆ 46     │
│ 복수매체        ┆ MULTI_TARGET     ┆ 14     │
└─────────────────┴──────────────────┴────────┘


### 행동 유형 파생 컬럼 생성

In [19]:
# 텍스트 준비
df_list = df_list.with_columns(
    pl.concat_str([
        pl.col('ads_name').fill_null(''),
        pl.lit(' '),
        pl.col('ads_summary').fill_null(''),
        pl.lit(' '),
        pl.col('ads_guide').fill_null('')
    ]).str.to_lowercase().alias('action_text')
)

# 행동 키워드 flag
df_list = df_list.with_columns([
    pl.col('action_text').str.contains(r'설치|다운로드|다운\b|앱설치').cast(pl.Int8).alias('action_install_target'),
    pl.col('action_text').str.contains(r'실행|방문|접속|이동|오픈|클릭|바로가기|둘러보기|구경').cast(pl.Int8).alias('action_visit_target'),
    pl.col('action_text').str.contains(r'팔로우|구독|좋아요|회원가입|가입|퀴즈|문제풀|설문|사전예약|체험|저장|참여|응모|인증|리뷰작성|후기작성').cast(pl.Int8).alias('action_engage_target'),
    pl.col('action_text').str.contains(r'구매|결제|주문|구입').cast(pl.Int8).alias('action_buy_target'),
])

df_list = df_list.with_columns([
    (
        pl.col('action_install_target') + pl.col('action_visit_target') +
        pl.col('action_engage_target') + pl.col('action_buy_target')
    ).alias('action_target_cnt')
])

In [20]:
# 행동 유형 값 체계 통일 (자연어·숫자 혼재 → 전부 숫자 코드)
# 코드 의미: 0=미확정, 1=설치, 2=방문, 3=참여, 12=구매, 99=복합행동
df_list = df_list.with_columns([
    pl.when(pl.col('action_target_cnt') >= 2).then(pl.lit('99'))
    .when(pl.col('action_buy_target') == 1).then(pl.lit('12'))
    .when(pl.col('action_install_target') == 1).then(pl.lit('1'))
    .when(pl.col('action_engage_target') == 1).then(pl.lit('3'))
    .when(pl.col('action_visit_target') == 1).then(pl.lit('2'))
    .otherwise(pl.lit('0'))
    .cast(pl.Categorical)
    .alias('ads_action_rule'),

    pl.col('ads_type').cast(pl.Utf8).alias('ads_type_raw'),
])

# action_confidence를 먼저 정의한 뒤 ads_action_diff_flag에서 참조
# (기존: 같은 with_columns 안에서 동시 정의 → ColumnNotFoundError 발생)
df_list = df_list.with_columns([
    pl.when(pl.col('action_target_cnt') == 1).then(pl.lit('HIGH'))
    .when(pl.col('action_target_cnt') >= 2).then(pl.lit('MULTI_ACTION'))
    .otherwise(pl.lit('NO_ACTION_MATCH'))
    .alias('action_confidence'),
])

df_list = df_list.with_columns([
    pl.when(
        (pl.col('action_confidence') == 'HIGH') &
        (pl.col('ads_type').cast(pl.Utf8) != pl.col('ads_action_rule').cast(pl.Utf8)) &
        (~pl.col('ads_action_rule').cast(pl.Utf8).is_in(['0', '99']))
    ).then(pl.lit(1)).otherwise(pl.lit(0))
    .cast(pl.Int8).alias('ads_action_diff_flag')
])

print('=== 행동 키워드 히트 수 ===')
print(df_list.select([
    pl.col('action_install_target').sum().alias('install_rows'),
    pl.col('action_visit_target').sum().alias('visit_rows'),
    pl.col('action_engage_target').sum().alias('engage_rows'),
    pl.col('action_buy_target').sum().alias('buy_rows'),
]))

print('\n=== ads_action_rule 분포 (0=미확정, 1=설치, 2=방문, 3=참여, 12=구매, 99=복합행동) ===')
print(df_list['ads_action_rule'].value_counts().sort('count', descending=True))

print('\n=== action_confidence 분포 ===')
print(df_list['action_confidence'].value_counts().sort('count', descending=True))

=== 행동 키워드 히트 수 ===
shape: (1, 4)
┌──────────────┬────────────┬─────────────┬──────────┐
│ install_rows ┆ visit_rows ┆ engage_rows ┆ buy_rows │
│ ---          ┆ ---        ┆ ---         ┆ ---      │
│ i64          ┆ i64        ┆ i64         ┆ i64      │
╞══════════════╪════════════╪═════════════╪══════════╡
│ 4668         ┆ 383458     ┆ 445076      ┆ 13696    │
└──────────────┴────────────┴─────────────┴──────────┘

=== ads_action_rule 분포 (0=미확정, 1=설치, 2=방문, 3=참여, 12=구매, 99=복합행동) ===
shape: (6, 2)
┌─────────────────┬────────┐
│ ads_action_rule ┆ count  │
│ ---             ┆ ---    │
│ cat             ┆ u32    │
╞═════════════════╪════════╡
│ 99              ┆ 390191 │
│ 3               ┆ 54885  │
│ 0               ┆ 154    │
│ 2               ┆ 19     │
│ 12              ┆ 7      │
│ 1               ┆ 4      │
└─────────────────┴────────┘

=== action_confidence 분포 ===
shape: (3, 2)
┌───────────────────┬────────┐
│ action_confidence ┆ count  │
│ ---               ┆ ---    │
│ str       

---
## 3. 유저 일자 활동 테이블

### 테이블 설명
- dvc_idx + active_date + mda_idx = 1행 (유저 x 날짜 x 매체)

### 전처리 내용
1. first_click, last_click String → datetime 형변환 + 파싱 실패 검증
2. 시간/요일 파생 컬럼 생성
3. is_converted : complete_cnt 기반으로 재정의
4. 반복 참여 패턴 분석용 지표 추가
5. 그레인 검증

In [21]:
df_user_activity = pl.read_parquet(DATA_PATH + 'tb_user_daily_activity.parquet')
print('shape:', df_user_activity.shape)

shape: (1610768, 12)


In [22]:
# 타입 확인
print(df_user_activity['active_date'].dtype)
print(df_user_activity['first_click'].dtype)
print(df_user_activity['last_click'].dtype)

# active_date, first_click, last_click 모두 타입 체크 후 변환
df_user_activity = df_user_activity.with_columns([
    pl.col('active_date').str.to_date(strict=False)
    if df_user_activity['active_date'].dtype in (pl.Utf8, pl.String)
    else pl.col('active_date'),

    pl.col('first_click').str.to_datetime(strict=False)
    if df_user_activity['first_click'].dtype in (pl.Utf8, pl.String)
    else pl.col('first_click'),

    pl.col('last_click').str.to_datetime(strict=False)
    if df_user_activity['last_click'].dtype in (pl.Utf8, pl.String)
    else pl.col('last_click'),
])

# 파싱 실패 검증
check_parse_nulls(df_user_activity, ['active_date', 'first_click', 'last_click'], 'df_user_activity')

# 시간 파생 컬럼 생성
df_user_activity = df_user_activity.with_columns([
    pl.col('active_date').dt.weekday().alias('weekday'),
    pl.col('first_click').dt.hour().alias('start_hour'),
    pl.col('first_click').dt.strftime('%H:%M:%S').alias('start_time'),
    pl.col('last_click').dt.hour().alias('end_hour'),
    pl.col('last_click').dt.strftime('%H:%M:%S').alias('end_time'),
])
df_user_activity = df_user_activity.with_columns(
    pl.col('weekday').cast(pl.String).replace(weekday_map_int).alias('weekday')
)

df_user_activity = df_user_activity.with_columns(
    (pl.col('complete_cnt') > 0).cast(pl.Int8).alias('is_converted')
)

print('shape:', df_user_activity.shape)
print('\n전환 여부 분포:')
print(df_user_activity['is_converted'].value_counts())

String
String
String

── [df_user_activity] 날짜 파싱 null 검증 ──
  ✅  active_date: null 0건
  ✅  first_click: null 0건
  ✅  last_click: null 0건
shape: (1610768, 18)

전환 여부 분포:
shape: (2, 2)
┌──────────────┬─────────┐
│ is_converted ┆ count   │
│ ---          ┆ ---     │
│ i8           ┆ u32     │
╞══════════════╪═════════╡
│ 1            ┆ 1005042 │
│ 0            ┆ 605726  │
└──────────────┴─────────┘


In [23]:
# 반복 참여 패턴 분석을 위한 지표 추가
# 유저-날짜 단위로 매체/광고 활동을 통합하되,
# 앞 셀에서 만든 파생 컬럼(first_click, last_click, start_hour, weekday 등)이
# 최종 user_daily_activity_clean.parquet에 빠지지 않도록 함께 집계한다.

user_daily_agg_exprs = [
    pl.sum("click_cnt").alias("click_cnt"),
    pl.sum("complete_cnt").alias("complete_cnt"),
]

# 유저가 하루 동안 접촉한 매체/광고 수
if "mda_idx" in df_user_activity.columns:
    user_daily_agg_exprs.append(pl.n_unique("mda_idx").alias("unique_media_cnt"))

if "ads_idx" in df_user_activity.columns:
    user_daily_agg_exprs.append(pl.n_unique("ads_idx").alias("unique_ads_cnt"))

# 하루의 첫 클릭 / 마지막 클릭
if "first_click" in df_user_activity.columns:
    user_daily_agg_exprs.append(pl.min("first_click").alias("first_click"))

if "last_click" in df_user_activity.columns:
    user_daily_agg_exprs.append(pl.max("last_click").alias("last_click"))

# 앞 셀에서 만든 시간 파생 컬럼 보존
if "start_hour" in df_user_activity.columns:
    user_daily_agg_exprs.append(pl.min("start_hour").alias("start_hour"))

if "start_time" in df_user_activity.columns:
    user_daily_agg_exprs.append(pl.min("start_time").alias("start_time"))

if "end_hour" in df_user_activity.columns:
    user_daily_agg_exprs.append(pl.max("end_hour").alias("end_hour"))

if "end_time" in df_user_activity.columns:
    user_daily_agg_exprs.append(pl.max("end_time").alias("end_time"))

if "weekday" in df_user_activity.columns:
    user_daily_agg_exprs.append(pl.first("weekday").alias("weekday"))

# ctit 요약값이 원본 활동 테이블에 있으면 같이 보존
if "mean_ctit" in df_user_activity.columns:
    user_daily_agg_exprs.append(pl.mean("mean_ctit").alias("mean_ctit"))

if "median_ctit" in df_user_activity.columns:
    user_daily_agg_exprs.append(pl.median("median_ctit").alias("median_ctit"))

# 유저-날짜 단위 집계
user_daily = (
    df_user_activity
    .group_by(["dvc_idx", "active_date"])
    .agg(user_daily_agg_exprs)
)

user_daily = user_daily.sort(["dvc_idx", "active_date"]).with_columns([
    # 누적 방문 일수
    pl.col("active_date")
      .cum_count()
      .over("dvc_idx")
      .alias("active_days_cnt"),

    # 재방문 주기 (이전 방문일과의 차이, 첫 방문은 -1)
    pl.col("active_date")
      .diff()
      .over("dvc_idx")
      .dt.total_days()
      .fill_null(-1)
      .alias("days_since_last_click"),

    # 전환 여부
    (pl.col("complete_cnt") > 0)
      .cast(pl.Int8)
      .alias("is_converted"),

    # 일별 전환율
    pl.when(pl.col("click_cnt") > 0)
      .then(pl.col("complete_cnt") / pl.col("click_cnt"))
      .otherwise(0)
      .alias("daily_conversion_rate"),

    # 최초 방문일
    pl.col("active_date")
      .min()
      .over("dvc_idx")
      .alias("first_date"),

    # 최초 방문일로부터 경과된 일수
    (
        pl.col("active_date") - pl.col("active_date").min().over("dvc_idx")
    ).dt.total_days().alias("day_n"),
])

user_daily = user_daily.sort(["dvc_idx", "active_date"]).with_columns([
    # D1 리텐션: 전날 방문 후 다음날 재방문
    (pl.col("days_since_last_click") == 1)
      .cast(pl.Int8)
      .alias("is_d1_retention"),

    # D2 리텐션
    (pl.col("days_since_last_click") == 2)
      .cast(pl.Int8)
      .alias("is_d2_retention"),

    # D3 리텐션
    (pl.col("days_since_last_click") == 3)
      .cast(pl.Int8)
      .alias("is_d3_retention"),

    # D7+ 리텐션: 7일 이상 만에 재방문
    (pl.col("days_since_last_click") >= 7)
      .cast(pl.Int8)
      .alias("is_d7_retention"),

    # 이탈 위험: 3일 이상 공백 후 재방문
    (pl.col("days_since_last_click") >= 3)
      .cast(pl.Int8)
      .alias("is_churn_risk"),
])

# 그레인 검증
dup_user_daily = (
    user_daily.group_by(["dvc_idx", "active_date"]).len()
    .filter(pl.col("len") > 1).height
)

print(f"[그레인 검증] user_daily 중복행 수 (dvc_idx+active_date): {dup_user_daily}")
print("user_daily shape:", user_daily.shape)
print("user_daily columns:")
print(user_daily.columns)

print("\n리텐션 플래그 분포:")
for col in ["is_d1_retention", "is_d2_retention", "is_d3_retention",
            "is_d7_retention", "is_churn_risk"]:
    cnt = user_daily[col].sum()
    print(f"  {col}: {cnt:,}건")


[그레인 검증] user_daily 중복행 수 (dvc_idx+active_date): 0
user_daily shape: (1554523, 25)
user_daily columns:
['dvc_idx', 'active_date', 'click_cnt', 'complete_cnt', 'unique_media_cnt', 'first_click', 'last_click', 'start_hour', 'start_time', 'end_hour', 'end_time', 'weekday', 'mean_ctit', 'median_ctit', 'active_days_cnt', 'days_since_last_click', 'is_converted', 'daily_conversion_rate', 'first_date', 'day_n', 'is_d1_retention', 'is_d2_retention', 'is_d3_retention', 'is_d7_retention', 'is_churn_risk']

리텐션 플래그 분포:
  is_d1_retention: 186,081건
  is_d2_retention: 91,707건
  is_d3_retention: 53,069건
  is_d7_retention: 226,540건
  is_churn_risk: 389,153건


---
## 4. 광고 운영 캘린더 테이블

### 테이블 설명
- ads_idx + click_date = 1행 (광고 x 날짜)

### 전처리 내용
1. 형변환 + 요일 파생 컬럼 생성
2.  campaign_n_day 음수 직접 삭제 → `flag_invalid_campaign_n_day` 플래그화
3. 분석용 테이블 `df_sche_analysis` 분리 (원본 `df_sche` 유지)
4. 그레인 검증

In [24]:
df_sche = pl.read_parquet(DATA_PATH + 'sched.parquet')
print('shape:', df_sche.shape)

shape: (473067, 12)


In [25]:
# 형변환
df_sche = df_sche.with_columns([
    pl.col('ads_type').cast(pl.Utf8).cast(pl.Categorical),
    pl.col('ads_category').cast(pl.Utf8).cast(pl.Categorical),
    pl.when(pl.col('ads_day_cap') == 'Y').then(True)
      .when(pl.col('ads_day_cap') == 'N').then(False)
      .otherwise(None).alias('ads_day_cap'),
])

# ads_sdate, ads_edate datetime 변환
# String이면 파싱, 이미 datetime이면 그대로 통과
def to_datetime_safe(df, col):
    if df[col].dtype in (pl.Utf8, pl.String):
        return pl.col(col).str.strptime(pl.Datetime, format='%Y-%m-%d %H:%M:%S', strict=False)
    return pl.col(col)  # 이미 datetime이면 그대로

df_sche = df_sche.with_columns([
    to_datetime_safe(df_sche, 'ads_sdate').alias('ads_sdate'),
    to_datetime_safe(df_sche, 'ads_edate').alias('ads_edate'),
])

# 파싱 실패 검증
check_parse_nulls(df_sche, ['ads_sdate', 'ads_edate'], 'df_sche')
df_sche = df_sche.with_columns(
    pl.col('click_date').dt.weekday().cast(pl.String).replace(weekday_map_str).alias('click_weekday')
)

# 직접 삭제 대신 플래그 세우기
# campaign_n_day 음수 → flag_invalid_campaign_n_day 플래그 추가
# 원본 df_sche는 유지, df_sche_analysis에서만 필터링
df_sche = df_sche.with_columns([
    pl.col('click_date').is_not_null().cast(pl.Int8).alias('is_click'),
    pl.col('ads_sdate').is_null().cast(pl.Int8).alias('is_ads_sdate_zero'),
    pl.col('campaign_n_day').is_null().cast(pl.Int8).alias('is_campaign_n_day_null'),
    (
        pl.col('campaign_n_day').is_not_null() & (pl.col('campaign_n_day') < 0)
    ).cast(pl.Int8).alias('flag_invalid_campaign_n_day'),
])

invalid_cnt = int(df_sche['flag_invalid_campaign_n_day'].sum())
print(f'campaign_n_day 음수 플래그 건수: {invalid_cnt} (원본은 유지됨)')

# 분석용 테이블 분리
df_sche_analysis = df_sche.filter(pl.col('flag_invalid_campaign_n_day') == 0)
print(f'분석용 df_sche_analysis shape: {df_sche_analysis.shape}')
print('\n플래그 분포:')
print(df_sche.select(['is_click', 'is_ads_sdate_zero', 'is_campaign_n_day_null', 'flag_invalid_campaign_n_day']).sum())

# 그레인 검증
dup_sche = (
    df_sche.group_by(['ads_idx', 'click_date']).len()
    .filter(pl.col('len') > 1).height
)
print(f'\n[그레인 검증] df_sche 중복행 수 (ads_idx+click_date): {dup_sche}')


── [df_sche] 날짜 파싱 null 검증 ──
  ⚠️  주의!  ads_sdate: null 391건
  ⚠️  주의!  ads_edate: null 391건
campaign_n_day 음수 플래그 건수: 1 (원본은 유지됨)
분석용 df_sche_analysis shape: (473066, 17)

플래그 분포:
shape: (1, 4)
┌──────────┬───────────────────┬────────────────────────┬─────────────────────────────┐
│ is_click ┆ is_ads_sdate_zero ┆ is_campaign_n_day_null ┆ flag_invalid_campaign_n_day │
│ ---      ┆ ---               ┆ ---                    ┆ ---                         │
│ i64      ┆ i64               ┆ i64                    ┆ i64                         │
╞══════════╪═══════════════════╪════════════════════════╪═════════════════════════════╡
│ 37324    ┆ 391               ┆ 435752                 ┆ 1                           │
└──────────┴───────────────────┴────────────────────────┴─────────────────────────────┘

[그레인 검증] df_sche 중복행 수 (ads_idx+click_date): 0


---
## 5. 광고성과 테이블

### 테이블 설명
- ads_idx = 1행 (광고 단위)

### 전처리 내용
1. 형변환
2. NULL 플래그 생성
3. ads_order 노출 구간 삼분화

In [26]:
df_ads_outcome = pl.read_parquet(DATA_PATH + 'ads_outcome.parquet')
print('shape:', df_ads_outcome.shape)

shape: (445260, 11)


In [27]:
# 형변환
df_ads_outcome = df_ads_outcome.with_columns([
    pl.col('ads_type').cast(pl.String).cast(pl.Categorical),
    pl.col('ads_category').cast(pl.String).cast(pl.Categorical),
    pl.col('ads_rejoin_type').cast(pl.Categorical),
])

# NULL 플래그 생성
# avg_ctit NULL = complete_cnt 0인 광고 (클릭은 있어도 적립 완료 없으면 ctit 계산 불가)
df_ads_outcome = df_ads_outcome.with_columns([
    pl.when(pl.col('avg_ctit').is_null()).then(1).otherwise(0).cast(pl.Int8).alias('is_avg_ctit_null'),
    pl.when(pl.col('total_reward_cost').is_null()).then(1).otherwise(0).cast(pl.Int8).alias('is_total_reward_cost_null'),
])

# ads_order 노출 구간 삼분화
p33 = df_ads_outcome['ads_order'].quantile(0.33)
p67 = df_ads_outcome['ads_order'].quantile(0.67)

df_ads_outcome = df_ads_outcome.with_columns([
    pl.when(pl.col('ads_order') >= p67).then(pl.lit('Y')).otherwise(pl.lit('N')).alias('is_top_exposure'),
    pl.when(pl.col('ads_order') >= p67).then(pl.lit('상위'))
    .when(pl.col('ads_order') >= p33).then(pl.lit('중위'))
    .otherwise(pl.lit('하위'))
    .alias('exposure_grade'),
])

print('노출 구간 분포:')
print(df_ads_outcome['exposure_grade'].value_counts())
print('\nNULL 플래그 합계:')
print(df_ads_outcome.select(['is_avg_ctit_null', 'is_total_reward_cost_null']).sum())

노출 구간 분포:
shape: (3, 2)
┌────────────────┬────────┐
│ exposure_grade ┆ count  │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ 하위           ┆ 146911 │
│ 중위           ┆ 147295 │
│ 상위           ┆ 151054 │
└────────────────┴────────┘

NULL 플래그 합계:
shape: (1, 2)
┌──────────────────┬───────────────────────────┐
│ is_avg_ctit_null ┆ is_total_reward_cost_null │
│ ---              ┆ ---                       │
│ i64              ┆ i64                       │
╞══════════════════╪═══════════════════════════╡
│ 436881           ┆ 435743                    │
└──────────────────┴───────────────────────────┘


---
## 6. 분석 테이블

### 테이블 설명
- `main_funnel` : 클릭 1건 = 1행. 퍼널 분석 기본 테이블
- `fact_click_reward` : 클릭-적립 연결 팩트 테이블

### 전처리 내용
1. 날짜 형변환 + **[피드백 1]** 파싱 실패 검증
2. Categorical / Boolean 형변환
- `df_ji_labeled`: `ads_join_info_labeled.parquet`에서 읽는 광고참여목록 정제 테이블. 클릭/완료 기준 파생 날짜 컬럼을 보존해 EDA-ready에 저장한다.



In [28]:
df_main = pl.read_parquet(DATA_PATH + 'main_funnel.parquet')
df_fact  = pl.read_parquet(PARQUET_PATH + 'IVE_광고적립__Result *.parquet')

# 전처리파트 다듬기 파일에만 있던 광고참여목록 정제 테이블
# click_key 기준 클릭/참여 로그 성격의 팩트 테이블로, EDA-ready 산출물에도 별도 저장한다.
df_ji_labeled = pl.read_parquet(DATA_PATH + 'ads_join_info_labeled.parquet')

print('main_funnel shape          :', df_main.shape)
print('fact_click_reward shape    :', df_fact.shape)
print('ads_join_info_labeled shape:', df_ji_labeled.shape)


main_funnel shape          : (3181777, 39)
fact_click_reward shape    : (1475031, 15)
ads_join_info_labeled shape: (16854865, 33)


In [29]:
# 타입 체크 후 변환 (이미 date/datetime이면 그대로)
def cast_if_str(df, col, fmt=None, to='date'):
    if col not in df.columns:
        return None

    if df[col].dtype in (pl.Utf8, pl.String):
        if to == 'date':
            return pl.col(col).str.replace_all('"', '').str.to_date(format=fmt, strict=False)
        else:
            return pl.col(col).str.replace_all('"', '').str.to_datetime(format=fmt, strict=False)
    return pl.col(col)


def apply_existing_exprs(df, exprs):
    # None이 아닌 expression만 적용한다.
    valid_exprs = [expr for expr in exprs if expr is not None]
    if valid_exprs:
        return df.with_columns(valid_exprs)
    return df


# main_funnel 형변환
main_exprs = [
    cast_if_str(df_main, 'click_day',  '%Y-%m-%d', 'date'),
    pl.col('click_time').cast(pl.Int8) if 'click_time' in df_main.columns else None,
    cast_if_str(df_main, 'click_date', '%Y-%m-%d %H:%M:%S', 'datetime'),
    cast_if_str(df_main, 'exp_day',    '%Y-%m-%d', 'date'),
    pl.col('ads_type').cast(pl.Utf8).cast(pl.Categorical) if 'ads_type' in df_main.columns else None,
    pl.col('ads_category').cast(pl.Utf8).cast(pl.Categorical) if 'ads_category' in df_main.columns else None,
    pl.col('ads_rejoin_type').cast(pl.Categorical) if 'ads_rejoin_type' in df_main.columns else None,
    pl.col('is_completed').cast(pl.Boolean) if 'is_completed' in df_main.columns else None,
]
df_main = apply_existing_exprs(df_main, main_exprs)


# fact_click_reward 형변환 (IVE_광고적립 실제 컬럼 기준)
fact_exprs = [
    cast_if_str(df_fact, 'click_date', '%Y-%m-%d %H:%M:%S', 'datetime'),
    cast_if_str(df_fact, 'regdate',    '%Y-%m-%d %H:%M:%S', 'datetime'),
]
df_fact = apply_existing_exprs(df_fact, fact_exprs)

# click_day 파생 컬럼 생성 (click_date → date만 추출)
if 'click_date' in df_fact.columns:
    df_fact = df_fact.with_columns(
        pl.col('click_date').dt.date().alias('click_day')
    )

# 있는 컬럼만 추가 변환
for col, expr in [
    ('click_time',           pl.col('click_time').cast(pl.Int8)),
    ('is_completed',         pl.col('is_completed').cast(pl.Boolean)),
    ('is_delayed_complete',  pl.col('is_delayed_complete').cast(pl.Boolean)),
    ('is_same_day_complete', pl.col('is_same_day_complete').cast(pl.Boolean)),
]:
    if col in df_fact.columns:
        df_fact = df_fact.with_columns(expr)

# margin 컬럼 생성 (없는 경우)
if ('margin' not in df_fact.columns) and {'adv_cost', 'earn_cost'}.issubset(set(df_fact.columns)):
    df_fact = df_fact.with_columns(
        (pl.col('adv_cost') - pl.col('earn_cost')).alias('margin')
    )


# ads_join_info_labeled 형변환
ji_exprs = [
    pl.col('click_time').cast(pl.Int8) if 'click_time' in df_ji_labeled.columns else None,
    cast_if_str(df_ji_labeled, 'click_date', '%Y-%m-%d %H:%M:%S', 'datetime'),
    cast_if_str(df_ji_labeled, 'regdate',    '%Y-%m-%d %H:%M:%S', 'datetime'),
    pl.col('is_completed').cast(pl.Boolean) if 'is_completed' in df_ji_labeled.columns else None,
    pl.col('is_delayed_complete').cast(pl.Boolean) if 'is_delayed_complete' in df_ji_labeled.columns else None,
    pl.col('is_same_day_complete').cast(pl.Boolean) if 'is_same_day_complete' in df_ji_labeled.columns else None,
]
df_ji_labeled = apply_existing_exprs(df_ji_labeled, ji_exprs)

# margin 컬럼 생성 (없는 경우)
if ('margin' not in df_ji_labeled.columns) and {'adv_cost', 'earn_cost'}.issubset(set(df_ji_labeled.columns)):
    df_ji_labeled = df_ji_labeled.with_columns(
        (pl.col('adv_cost') - pl.col('earn_cost')).alias('margin')
    )

# 날짜/시간 파생 컬럼 보장
# 원본에 이미 있으면 같은 이름으로 다시 계산해 타입을 맞춘다.
ji_time_exprs = []
if 'click_date' in df_ji_labeled.columns:
    ji_time_exprs += [
        pl.col('click_date').dt.date().alias('click_date_only'),
        pl.col('click_date').dt.hour().cast(pl.Int8).alias('click_hour'),
        pl.col('click_date').dt.weekday().cast(pl.String).replace(weekday_map_str).alias('click_weekday'),
    ]
if 'regdate' in df_ji_labeled.columns:
    ji_time_exprs += [
        pl.col('regdate').dt.date().alias('reg_date_only'),
        pl.col('regdate').dt.hour().cast(pl.Int8).alias('reg_hour'),
        pl.col('regdate').dt.weekday().cast(pl.String).replace(weekday_map_str).alias('reg_weekday'),
    ]
if ji_time_exprs:
    df_ji_labeled = df_ji_labeled.with_columns(ji_time_exprs)

# 파싱 실패 검증
check_parse_nulls(df_main, ['click_date'], 'df_main')
check_parse_nulls(df_fact, ['click_date', 'regdate'], 'df_fact')
check_parse_nulls(df_ji_labeled, ['click_date', 'regdate'], 'df_ji_labeled')

print('전처리 완료')
print('main_funnel shape          :', df_main.shape)
print('fact_click_reward shape    :', df_fact.shape)
print('ads_join_info_labeled shape:', df_ji_labeled.shape)
print('df_ji_labeled columns:')
print(df_ji_labeled.columns)



── [df_main] 날짜 파싱 null 검증 ──
  ✅  click_date: null 0건

── [df_fact] 날짜 파싱 null 검증 ──
  ✅  click_date: null 0건
  ✅  regdate: null 0건

── [df_ji_labeled] 날짜 파싱 null 검증 ──
  ✅  click_date: null 0건
  ⚠️  주의!  regdate: null 15,379,915건
전처리 완료
main_funnel shape          : (3181777, 39)
fact_click_reward shape    : (1475031, 17)
ads_join_info_labeled shape: (16854865, 39)
df_ji_labeled columns:
['click_key', 'ads_idx', 'dvc_idx', 'mda_idx', 'pub_sub_rel_id', 'adv_price', 'contract_price', 'media_price', 'reward_price', 'reward_point', 'click_day', 'click_time', 'click_date', 'exp_day', 'network', 'carrier', 'user_ip', 'click_day_1', 'rwd_idx', 'regdate', 'ctit', 'show_cost', 'adv_cost', 'earn_cost', 'rwd_cost', 'is_completed', 'reg_day', 'is_same_day_complete', 'is_delayed_complete', 'margin', 'is_anomaly_day', 'remove_cond1_flag', 'remove_cond2_flag', 'click_date_only', 'click_hour', 'click_weekday', 'reg_date_only', 'reg_hour', 'reg_weekday']


### df_main 추가 전처리
- 날짜 컬럼 추가 변환 (reg_day, regdate)
- 불필요 컬럼 제거 (network, carrier, user_ip, click_day_1)
- 재무 이상치 플래그 생성 (show_cost, adv_cost)

In [30]:
# reg_day, regdate 날짜 추가 변환
# (cast_if_str 함수로 타입 체크 후 변환)
df_main = df_main.with_columns([
    cast_if_str(df_main, 'reg_day',  '%Y-%m-%d',          'date'),
    cast_if_str(df_main, 'regdate',  '%Y-%m-%d %H:%M:%S', 'datetime'),
])

# 파싱 실패 검증
check_parse_nulls(df_main, ['reg_day', 'regdate'], 'df_main (reg_day/regdate)')
print("reg_day, regdate 변환 완료")



── [df_main (reg_day/regdate)] 날짜 파싱 null 검증 ──
  ⚠️  주의!  reg_day: null 1,797,993건
  ⚠️  주의!  regdate: null 1,797,993건
reg_day, regdate 변환 완료


In [31]:
# 결측이 너무 많거나 분석에 불필요한 컬럼 제거
# - network, carrier, user_ip: 결측 과다 → 분석 기여도 낮음
# - click_day_1: join 과정에서 생긴 중복 컬럼

drop_cols = [c for c in ["network", "carrier", "user_ip", "click_day_1"]
             if c in df_main.columns]

if drop_cols:
    df_main = df_main.drop(drop_cols)
    print(f"제거된 컬럼: {drop_cols}")
else:
    print("제거할 컬럼 없음 (이미 없거나 컬럼명 다름)")

print("df_main shape after drop:", df_main.shape)


제거된 컬럼: ['network', 'carrier', 'user_ip', 'click_day_1']
df_main shape after drop: (3181777, 35)


In [32]:
# 재무 이상치 플래그 생성
# - show_cost가 0 또는 1인 경우: 광고 노출 비용이 비정상적으로 낮음
# - adv_cost가 0인 경우: 광고주 단가가 0 → 수익 발생 불가
# - is_finance_anomaly: 두 조건 중 하나라도 해당하면 True

df_main = df_main.with_columns([
    pl.col("show_cost")
      .is_in([0, 1])
      .alias("flag_show_cost_low"),

    (pl.col("adv_cost") == 0)
      .alias("flag_adv_cost_zero"),
]).with_columns(
    (pl.col("flag_show_cost_low") | pl.col("flag_adv_cost_zero"))
      .alias("is_finance_anomaly")
)

print("재무 이상치 플래그 분포:")
print("  flag_show_cost_low  :", df_main["flag_show_cost_low"].sum(),  "건")
print("  flag_adv_cost_zero  :", df_main["flag_adv_cost_zero"].sum(),  "건")
print("  is_finance_anomaly  :", df_main["is_finance_anomaly"].sum(),  "건")
print("  is_finance_anomaly 비율: {:.2f}%".format(
    df_main["is_finance_anomaly"].sum() / df_main.height * 100
))


재무 이상치 플래그 분포:
  flag_show_cost_low  : 21889 건
  flag_adv_cost_zero  : 15774 건
  is_finance_anomaly  : 21889 건
  is_finance_anomaly 비율: 0.69%


### 이상치 제거
- 광고 × 날짜별 클릭수가 비정상적으로 많은 경우 anomaly 판정 (IQR 상단 기준)
- `margin` = `adv_cost - earn_cost` 로 계산 후 join
- 제거 조건: `is_anomaly_day == 1` AND (`margin` is null OR `margin == 0`) — 음수 마진은 플래그로 별도 관리
- 원본 `df_fact` 유지, 정제 테이블 `df_fact_clean` 별도 생성
- `remove_cond2`: 완료율/재무 분석 시 수익 발생 행도 제거하는 조건 (별도 사용)

In [33]:
# 광고 x 날짜 클릭수 집계
daily_ads = (
    df_fact
    .group_by(['ads_idx', 'click_day'])
    .agg(pl.len().alias('clicks'))
)

# 광고별 IQR 기준 계산
daily_ads_flag = (
    daily_ads
    .join(
        daily_ads.group_by('ads_idx').agg(
            pl.col('clicks').quantile(0.25).alias('q1'),
            pl.col('clicks').quantile(0.75).alias('q3')
        ),
        on='ads_idx',
        how='left'
    )
    .with_columns(
        (pl.col('q3') - pl.col('q1')).alias('iqr')
    )
    .with_columns(
        (pl.col('q3') + pl.col('iqr') * 1.5).alias('upper_bound')
    )
    .with_columns(
        (pl.col('clicks') > pl.col('upper_bound'))
        .cast(pl.Int8)
        .alias('is_anomaly_day')
    )
)

print('anomaly_day 건수:', daily_ads_flag.filter(pl.col('is_anomaly_day') == 1).height)
print('전체 광고×날짜 건수:', daily_ads_flag.height)

anomaly_day 건수: 1233
전체 광고×날짜 건수: 26465


In [34]:
# df_fact에 is_anomaly_day 플래그 join
df_fact = df_fact.join(
    daily_ads_flag.select(['ads_idx', 'click_day', 'is_anomaly_day']),
    on=['ads_idx', 'click_day'],
    how='left'
).with_columns(
    pl.col('is_anomaly_day').fill_null(0)
)

# 제거 조건 정의
# remove_cond  : anomaly_day이면서 수익이 없는 행 (일반 분석용)
# remove_cond2 : anomaly_day인 행 전체 제거 (완료율/재무 분석용)
remove_cond = (
    (pl.col('is_anomaly_day') == 1) &
    (
        pl.col('margin').is_null() |
        (pl.col('margin') <= 0)
    )
)

remove_cond2 = (pl.col('is_anomaly_day') == 1)

# 정제 테이블 생성 (원본 df_fact 유지)
df_fact_clean  = df_fact.filter(~remove_cond)
df_fact_clean2 = df_fact.filter(~remove_cond2)

print(f'원본 df_fact      : {df_fact.shape[0]:,}행')
print(f'df_fact_clean     : {df_fact_clean.shape[0]:,}행  (remove_cond 적용)')
print(f'df_fact_clean2    : {df_fact_clean2.shape[0]:,}행  (remove_cond2 적용)')

원본 df_fact      : 1,475,031행
df_fact_clean     : 1,474,676행  (remove_cond 적용)
df_fact_clean2    : 1,305,493행  (remove_cond2 적용)


In [35]:
# df_main 기준으로 정제 테이블 생성
df_main_clean  = df_main.filter(pl.col('is_anomaly_day') == 0) if 'is_anomaly_day' in df_main.columns else df_main
df_main_clean2 = df_main.filter(pl.col('is_anomaly_day') == 0) if 'is_anomaly_day' in df_main.columns else df_main

# 원본 vs 정제 KPI 비교 (df_main 기준 - is_completed, margin 모두 보유)
def calc_kpi(df, label):
    return (
        df.select(
            pl.len().alias('clicks'),
            pl.col('is_completed').sum().alias('completed'),
            pl.col('margin').sum().alias('margin_sum')
        )
        .with_columns(
            (pl.col('completed') / pl.col('clicks') * 100).round(2).alias('cvr_pct'),
            pl.lit(label).alias('version')
        )
        .select(['version', 'clicks', 'completed', 'cvr_pct', 'margin_sum'])
    )

# remove_cond: anomaly_day이면서 margin이 null 또는 0인 행
remove_cond = (
    (pl.col('is_anomaly_day') == 1) &
    (
        pl.col('margin').is_null() |
        (pl.col('margin') == 0)   # 음수 마진은 플래그로 별도 관리
    )
)
remove_cond2 = (pl.col('is_anomaly_day') == 1)

df_fact_clean  = df_fact.filter(~remove_cond)  if 'is_anomaly_day' in df_fact.columns else df_fact
df_fact_clean2 = df_fact.filter(~remove_cond2) if 'is_anomaly_day' in df_fact.columns else df_fact

compare_kpi = pl.concat([
    calc_kpi(df_main,        '원본 테이블'),
    calc_kpi(df_main.filter(~remove_cond)  if 'is_anomaly_day' in df_main.columns else df_main, '정제(remove_cond)'),
    calc_kpi(df_main.filter(~remove_cond2) if 'is_anomaly_day' in df_main.columns else df_main, '정제(remove_cond2)'),
])
compare_kpi


version,clicks,completed,cvr_pct,margin_sum
str,u32,u32,f64,i64
"""원본 테이블""",3181777,1383784,43.49,171332229
"""정제(remove_cond)""",3181777,1383784,43.49,171332229
"""정제(remove_cond2)""",3181777,1383784,43.49,171332229


---
## 7. 전처리 완료 확인

shape 출력만 하던 것에서 아래 4가지 검증 추가:
- PK 중복 여부
- 핵심 컬럼 의도하지 않은 null 여부
- 음수가 나올 수 없는 컬럼 음수 여부
- 파생 플래그 분포

In [36]:
tables = {
    'finance_clean1':    finance_clean1,
    'finance_clean2':     finance_clean2,
    'finance_margin':   finance_margin,
    'df_rpt':           df_rpt,
    'df_join_info':     df_join_info,
    'df_rwd':           df_rwd,
    'df_list':          df_list,
    'df_user_activity': df_user_activity,
    'user_daily':       user_daily,
    'df_sche':          df_sche,
    'df_sche_analysis': df_sche_analysis,
    'df_ads_outcome':   df_ads_outcome,
    'df_main':          df_main,
    'df_ji_labeled':    df_ji_labeled,
    'df_fact':          df_fact,
    'df_fact_clean':    df_fact_clean,
    'df_fact_clean2':   df_fact_clean2,
}

print('=' * 60)
print('전처리 완료 테이블 목록')
print('=' * 60)
for name, tbl in tables.items():
    print(f'  {name:<20}: {tbl.shape}')


전처리 완료 테이블 목록
  finance_clean1      : (1474776, 28)
  finance_clean2      : (1383784, 27)
  finance_margin      : (1471248, 28)
  df_rpt              : (6953146, 10)
  df_join_info        : (16854865, 17)
  df_rwd              : (1475031, 15)
  df_list             : (445260, 68)
  df_user_activity    : (1610768, 18)
  user_daily          : (1554523, 25)
  df_sche             : (473067, 17)
  df_sche_analysis    : (473066, 17)
  df_ads_outcome      : (445260, 15)
  df_main             : (3181777, 38)
  df_ji_labeled       : (16854865, 39)
  df_fact             : (1475031, 18)
  df_fact_clean       : (1474857, 18)
  df_fact_clean2      : (1305493, 18)


In [37]:
# ── PK 중복 검증 ──
print('\n── PK 중복 검증 ──')
pk_checks = [
    ('df_sche',          ['ads_idx', 'click_date']),
    ('df_ads_outcome',   ['ads_idx']),
    ('df_user_activity', ['dvc_idx', 'active_date', 'mda_idx']),
    ('user_daily',       ['dvc_idx', 'active_date']),
    ('df_ji_labeled',    ['click_key']),
]
for tbl_name, pk_cols in pk_checks:
    tbl = tables[tbl_name]
    dup = tbl.group_by(pk_cols).len().filter(pl.col('len') > 1).height
    flag = '⚠️ ' if dup > 0 else '✅'
    print(f'  {flag} {tbl_name} ({"+".join(pk_cols)}): 중복 {dup}건')



── PK 중복 검증 ──
  ✅ df_sche (ads_idx+click_date): 중복 0건
  ✅ df_ads_outcome (ads_idx): 중복 0건
  ✅ df_user_activity (dvc_idx+active_date+mda_idx): 중복 0건
  ✅ user_daily (dvc_idx+active_date): 중복 0건
  ✅ df_ji_labeled (click_key): 중복 0건


In [38]:
# ── 핵심 컬럼 null 검증 ──
print('\n── 핵심 컬럼 null 검증 ──')
null_checks = [
    ('df_list',        ['ads_idx', 'ads_type', 'ads_name']),
    ('df_sche',        ['ads_idx', 'click_date']),
    ('df_ads_outcome', ['ads_idx', 'ads_order', 'click_cnt']),
    ('df_main',        ['ads_idx', 'click_date', 'is_completed']),
    ('df_ji_labeled',  ['click_key', 'ads_idx', 'click_date', 'is_completed']),
]
for tbl_name, cols in null_checks:
    tbl = tables[tbl_name]
    for col in cols:
        if col not in tbl.columns:
            continue
        n = tbl[col].is_null().sum()
        flag = '⚠️ ' if n > 0 else '✅'
        print(f'  {flag} {tbl_name}.{col}: null {n:,}건')



── 핵심 컬럼 null 검증 ──
  ✅ df_list.ads_idx: null 0건
  ✅ df_list.ads_type: null 0건
  ✅ df_list.ads_name: null 0건
  ✅ df_sche.ads_idx: null 0건
  ⚠️  df_sche.click_date: null 435,743건
  ✅ df_ads_outcome.ads_idx: null 0건
  ✅ df_ads_outcome.ads_order: null 0건
  ✅ df_ads_outcome.click_cnt: null 0건
  ✅ df_main.ads_idx: null 0건
  ✅ df_main.click_date: null 0건
  ✅ df_main.is_completed: null 0건
  ✅ df_ji_labeled.click_key: null 0건
  ✅ df_ji_labeled.ads_idx: null 0건
  ✅ df_ji_labeled.click_date: null 0건
  ✅ df_ji_labeled.is_completed: null 0건


In [39]:
# ── 음수 불가 컬럼 검증 ──
print('\n── 음수 불가 컬럼 검증 ──')
neg_checks = [
    ('df_ads_outcome', 'click_cnt'),
    ('df_ads_outcome', 'complete_cnt'),
    ('df_sche',        'campaign_n_day'),
    ('finance_clean1',   'adv_cost'),
    ('finance_clean1',   'earn_cost'),
    ('df_ji_labeled',    'adv_cost'),
    ('df_ji_labeled',    'earn_cost'),
    ('df_ji_labeled',    'rwd_cost'),
]
for tbl_name, col in neg_checks:
    tbl = tables[tbl_name]
    if col not in tbl.columns:
        continue
    n = tbl.filter(pl.col(col) < 0).height
    flag = '⚠️ ' if n > 0 else '✅'
    print(f'  {flag} {tbl_name}.{col}: 음수 {n:,}건')



── 음수 불가 컬럼 검증 ──
  ✅ df_ads_outcome.click_cnt: 음수 0건
  ✅ df_ads_outcome.complete_cnt: 음수 0건
  ⚠️  df_sche.campaign_n_day: 음수 1건
  ✅ finance_clean1.adv_cost: 음수 0건
  ✅ finance_clean1.earn_cost: 음수 0건
  ✅ df_ji_labeled.adv_cost: 음수 0건
  ✅ df_ji_labeled.earn_cost: 음수 0건
  ✅ df_ji_labeled.rwd_cost: 음수 0건


In [40]:
# ── 파생 플래그 분포 ──
print('\n── 파생 플래그 분포 ──')

print('\n  [finance_clean1] flag_special_margin_group:')
print(finance_clean1['flag_special_margin_group'].value_counts())

print('\n  [df_sche] flag_invalid_campaign_n_day:')
print(df_sche['flag_invalid_campaign_n_day'].value_counts())

print('\n  [df_ads_outcome] exposure_grade:')
print(df_ads_outcome['exposure_grade'].value_counts())

print('\n  [df_list] media_confidence:')
print(df_list['media_confidence'].value_counts().sort('count', descending=True))

print('\n  [df_list] ads_action_rule (0=미확정, 1=설치, 2=방문, 3=참여, 12=구매, 99=복합행동):')
print(df_list['ads_action_rule'].value_counts().sort('count', descending=True))

print('\n전처리 완료')


── 파생 플래그 분포 ──

  [finance_clean1] flag_special_margin_group:
shape: (2, 2)
┌───────────────────────────┬─────────┐
│ flag_special_margin_group ┆ count   │
│ ---                       ┆ ---     │
│ bool                      ┆ u32     │
╞═══════════════════════════╪═════════╡
│ false                     ┆ 1471248 │
│ true                      ┆ 3528    │
└───────────────────────────┴─────────┘

  [df_sche] flag_invalid_campaign_n_day:
shape: (2, 2)
┌─────────────────────────────┬────────┐
│ flag_invalid_campaign_n_day ┆ count  │
│ ---                         ┆ ---    │
│ i8                          ┆ u32    │
╞═════════════════════════════╪════════╡
│ 1                           ┆ 1      │
│ 0                           ┆ 473066 │
└─────────────────────────────┴────────┘

  [df_ads_outcome] exposure_grade:
shape: (3, 2)
┌────────────────┬────────┐
│ exposure_grade ┆ count  │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ 상위           ┆ 151054 │
│ 

---
## 8. EDA-ready 분석용 parquet 저장

이 파트는 EDA에서 바로 읽을 최소 분석용 parquet만 저장한다.  
핵심 원칙은 **각 테이블의 행 단위는 유지하되, 해당 테이블에서 이미 만든 파생 컬럼이 최종 parquet에 빠지지 않게 저장하는 것**이다.

저장 구조는 다음과 같다.

- `main_funnel.parquet`: 기존 `df_main` 기준. 이미 광고 코드값과 비용/마진 컬럼이 있으므로 별도 attr 조인 없이 라벨 파생컬럼만 추가한다. 기존에 만든 `flag_show_cost_low`, `flag_adv_cost_zero`, `is_finance_anomaly`, `is_anomaly_day`, `remove_cond1_flag`, `remove_cond2_flag` 등도 함께 저장한다.

- `ad_attr_map.parquet`: `df_list`에서 추출한 광고 단위 속성 지도이다. `ads_idx`를 기준으로 광고명, 기존 광고유형, 카테고리, 보상금액, 재참여 방식 등 광고 자체의 설명 정보를 관리하며, 클릭 수·완료 수·CVR·마진 등 성과 집계값은 포함하지 않는다. 향후 BERTopic 기반 재분류 결과가 확정되면 `ads_idx` 기준으로 새 광고유형 컬럼을 이 테이블에 추가하고, 필요 시 `ad_outcome.parquet` 또는 `main_funnel.parquet`와 머지하여 기존 분류와 재분류 기준의 성과 차이를 비교한다.

- `ad_outcome.parquet`: 현재 `df_ads_outcome`에 이미 광고 기본 속성이 포함되어 있으므로, 별도 조인 없이 라벨 파생컬럼과 `cvr_pct`, `is_valid_click10`만 추가한다. 기존에 만든 `is_avg_ctit_null`, `is_total_reward_cost_null`, `is_top_exposure`, `exposure_grade`도 함께 저장한다.

- `finance_clean1.parquet`, `finance_clean2.parquet`: 재무분석용 테이블이다. 별도 광고속성 조인은 하지 않고, 이미 만들어진 날짜/시간 파생 컬럼, 요일 컬럼, `ive_margin`, `flag_special_margin_group` 등을 포함해 저장한다.

- `sched_clean.parquet`: `df_sche_analysis` 기준. `is_click`, `is_ads_sdate_zero`, `is_campaign_n_day_null`, `flag_invalid_campaign_n_day`, `click_weekday` 등 운영 캘린더 파생 컬럼을 포함해 저장한다.

- `user_daily_activity_clean.parquet`: `user_daily` 기준. `df_user_activity`에서 만든 `first_click`, `last_click`, `start_hour`, `start_time`, `end_hour`, `end_time`, `weekday`를 유저-날짜 단위로 집계해 보존하고, 리텐션/전환 파생 컬럼도 포함해 저장한다.

- `hourly_report.parquet`, `ad_master_clean.parquet`: EDA에서 공통으로 읽을 기본 분석 테이블이다. `ad_master_clean.parquet`는 `df_list_for_eda`를 저장하므로 광고 라벨 및 텍스트 기반 파생 컬럼을 함께 포함한다.


In [41]:
# ============================================================
# EDA-ready 라벨 생성 보조 함수
# - action_type_5는 보관만 하고, 자동 우선 적용하지 않음
# - analysis_ads_type_label은 현재 기준에서는 ads_type_label 사용
# ============================================================

ADS_TYPE_LABEL_MAP = {
    1: "설치형",
    2: "실행형",
    3: "참여형",
    4: "클릭형",
    5: "제휴",
    6: "트위터",
    7: "인스타",
    8: "노출형",
    9: "퀘스트",
    10: "유튜브",
    11: "네이버",
    12: "CPS(물건구매)",
}

CATEGORY_LABEL_MAP = {
    0: "선택안함",
    1: "앱(간편적립)",
    2: "경험하기(CPI/CPE)",
    3: "구독(간편적립)",
    4: "간편미션-퀴즈",
    5: "경험하기(CPA)",
    6: "멀티보상(게임)",
    7: "금융(참여)",
    8: "무료참여",
    10: "유료참여",
    11: "쇼핑-상품별",
    12: "제휴몰",
    13: "간편미션",
}


def unique_keep_order(values):
    result = []

    for value in values:
        if value not in result:
            result.append(value)

    return result


def add_ad_label_columns(df):
    result = df.clone()

    # 1. ads_type → ads_type_label
    if "ads_type" in result.columns:
        result = result.with_columns(
            pl.col("ads_type")
            .cast(pl.String)
            .cast(pl.Int64, strict=False)
            .replace_strict(
                ADS_TYPE_LABEL_MAP,
                default=pl.lit("기타/미분류"),
                return_dtype=pl.String,
            )
            .alias("ads_type_label")
        )

    # 2. ads_category → category_name
    if "ads_category" in result.columns:
        result = result.with_columns(
            pl.col("ads_category")
            .cast(pl.String)
            .cast(pl.Int64, strict=False)
            .replace_strict(
                CATEGORY_LABEL_MAP,
                default=pl.lit("미분류/코드확인필요"),
                return_dtype=pl.String,
            )
            .alias("category_name")
        )

    # 3. ads_reward_price → reward_band
    if "ads_reward_price" in result.columns:
        reward_price = pl.col("ads_reward_price").cast(pl.Float64, strict=False)

        result = result.with_columns(
            pl.when(reward_price.is_null()).then(pl.lit("미상"))
            .when(reward_price < 50).then(pl.lit("50원 미만"))
            .when(reward_price < 100).then(pl.lit("50~99원"))
            .when(reward_price < 300).then(pl.lit("100~299원"))
            .when(reward_price < 500).then(pl.lit("300~499원"))
            .otherwise(pl.lit("500원 이상"))
            .alias("reward_band")
        )

    # 4. 최종 분석용 광고유형 기준
    # 현재는 기존 ads_type 기준 라벨을 사용한다.
    # action_type_5가 있어도 자동으로 덮어쓰지 않는다.
    if "ads_type_label" in result.columns:
        result = result.with_columns([
            pl.col("ads_type_label").alias("analysis_ads_type_label"),
            pl.lit("original_ads_type").alias("analysis_ads_type_source"),
        ])

    # 5. action_type_5 / action_subtype은 비교·검증용으로만 정리해서 보관
    if "action_type_5" in result.columns:
        result = result.with_columns(
            pl.col("action_type_5")
            .cast(pl.String)
            .str.strip_chars()
            .alias("action_type_5")
        )

    if "action_subtype" in result.columns:
        result = result.with_columns(
            pl.col("action_subtype")
            .cast(pl.String)
            .str.strip_chars()
            .alias("action_subtype")
        )

    return result


In [42]:
# ============================================================
# 8-1. ad_attr_map 생성
# ============================================================
# 순수 광고속성 지도만 만든다.
# click_cnt, complete_cnt, avg_ctit, total_reward_cost, margin 같은 성과/금액 집계값은 넣지 않는다.

df_list_for_eda = add_ad_label_columns(df_list)

ad_attr_cols = [
    "ads_idx",
    "ads_name",
    "ads_type",
    "ads_type_label",
    "analysis_ads_type_label",
    "analysis_ads_type_source",
    "ads_category",
    "category_name",
    "ads_reward_price",
    "reward_band",
    "ads_order",
    "ads_rejoin_type",
    "ads_save_way",
    "ads_day_cap",
    "ads_sdate",
    "ads_edate",
    "ads_action",
    "ads_action_rule",
    "ads_action_diff_flag",
    "action_type_5",
    "action_subtype",
    "bert_topic_id",
    "bert_topic_label",
    "bert_ads_type_label",
]

ad_attr_cols = [
    col for col in unique_keep_order(ad_attr_cols)
    if col in df_list_for_eda.columns
]

ad_attr_map = (
    df_list_for_eda
    .select(ad_attr_cols)
    .unique(subset=["ads_idx"], keep="first")
)

print("ad_attr_map shape:", ad_attr_map.shape)
print("ad_attr_map columns:")
print(ad_attr_map.columns)


ad_attr_map shape: (445260, 18)
ad_attr_map columns:
['ads_idx', 'ads_name', 'ads_type', 'ads_type_label', 'analysis_ads_type_label', 'analysis_ads_type_source', 'ads_category', 'category_name', 'ads_reward_price', 'reward_band', 'ads_order', 'ads_rejoin_type', 'ads_save_way', 'ads_day_cap', 'ads_sdate', 'ads_edate', 'ads_action_rule', 'ads_action_diff_flag']


In [43]:
# ============================================================
# 8-2. main_funnel / ad_outcome 생성
# ============================================================
# main_funnel은 이미 ads_type, ads_category, ads_reward_price 등이 있으므로
# 광고속성지도를 다시 조인하지 않고 라벨 파생컬럼만 추가한다.

main_funnel_for_eda = add_ad_label_columns(df_main)

# df_ads_outcome에도 이미 ads_name, ads_type, ads_category, ads_reward_price, ads_order, ads_rejoin_type이 있음.
# 따라서 ad_attr_map을 다시 조인하지 않고 라벨/파생 컬럼만 추가한다.
# 광고 집중도, 유형 성과, 카테고리 성과, 보상구간 성과 분석은 이 테이블을 사용한다.
ad_outcome = add_ad_label_columns(df_ads_outcome)

if ("click_cnt" in ad_outcome.columns) and ("complete_cnt" in ad_outcome.columns):
    ad_outcome = ad_outcome.with_columns([
        pl.when(pl.col("click_cnt") > 0)
        .then((pl.col("complete_cnt") / pl.col("click_cnt") * 100).round(2))
        .otherwise(pl.lit(None))
        .alias("cvr_pct"),

        (pl.col("click_cnt") >= 10)
        .cast(pl.Int8)
        .alias("is_valid_click10"),
    ])

print("main_funnel_for_eda shape:", main_funnel_for_eda.shape)
print("ad_outcome shape:", ad_outcome.shape)
print("ad_outcome columns:")
print(ad_outcome.columns)

main_funnel_for_eda shape: (3181777, 43)
ad_outcome shape: (445260, 22)
ad_outcome columns:
['ads_idx', 'ads_name', 'ads_type', 'ads_category', 'ads_reward_price', 'ads_order', 'ads_rejoin_type', 'click_cnt', 'complete_cnt', 'avg_ctit', 'total_reward_cost', 'is_avg_ctit_null', 'is_total_reward_cost_null', 'is_top_exposure', 'exposure_grade', 'ads_type_label', 'category_name', 'reward_band', 'analysis_ads_type_label', 'analysis_ads_type_source', 'cvr_pct', 'is_valid_click10']


In [44]:
# ============================================================
# 8-3. EDA-ready parquet 저장
# ============================================================
# EDA_READY_DIR은 0번 셀에서 이미 선언됨
EDA_READY_DIR.mkdir(parents=True, exist_ok=True)

print('EDA-ready 저장 폴더 준비 완료')
print('저장 경로:', EDA_READY_DIR)


# ------------------------------------------------------------
# 저장 직전 EDA용 테이블 정리
# ------------------------------------------------------------
finance_clean1_for_eda = add_ad_label_columns(finance_clean1)
finance_clean2_for_eda = add_ad_label_columns(finance_clean2)

sched_for_eda = df_sche_analysis if 'df_sche_analysis' in globals() else df_sche
sched_for_eda = add_ad_label_columns(sched_for_eda)

ad_master_clean_for_eda = df_list_for_eda

user_daily_activity_for_eda = user_daily


EDA-ready 저장 폴더 준비 완료
저장 경로: C:\Users\hoo58\OneDrive\바탕 화면\tables\DATA\eda_ready_new
